# Wine Cultivar Classification

Three cultivars of wine grown in the same region of Italy, each described by 13 chemical measurements — alcohol, malic acid, ash, alkalinity, magnesium, phenols, flavanoids and more. With only 178 wines, can chemistry alone tell which cultivar a wine came from?

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Wine (id 109) — distinct from Wine Quality (project 09), which is a
# different, larger dataset of Portuguese wines.
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/109/data.csv')))
print(df.shape)
print('Wines per cultivar code:')
print(df['class'].value_counts().sort_index())
df.head(3)


(178, 14)
Wines per cultivar code:
class
1    59
2    71
3    48
Name: count, dtype: int64


,Alcohol,Malicacid,Ash,Alcalinity_of_ash,Magnesium,Total_phenols,Flavanoids,Nonflavanoid_phenols,Proanthocyanins,Color_intensity,Hue,0D280_0D315_of_diluted_wines,Proline,class
0,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065,1
1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050,1
2,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185,1


In [2]:

feats = [c for c in df.columns if c != 'class']
X = df[feats]; y = df['class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 1.0000
Logistic Regression accuracy: 1.0000
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        15
           2       1.00      1.00      1.00        18
           3       1.00      1.00      1.00        12

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which chemicals separate the cultivars?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


Flavanoids                      0.168762
Color_intensity                 0.166957
Proline                         0.143297
Alcohol                         0.129551
Hue                             0.101328
0D280_0D315_of_diluted_wines    0.092843
Total_phenols                   0.046082
Proanthocyanins                 0.031669
Malicacid                       0.030680
Magnesium                       0.030426
Alcalinity_of_ash               0.027822
Nonflavanoid_phenols            0.015513
Ash                             0.015070
dtype: float64


Winner: Random Forest
